# J1.2 v3 — joint piecewise-SVF development search

Отчёт читает frozen search `3 variants × 3 development cases` на commit `fcfeb1c`. Exact-truth preflight до него прошёл 3/3. Held-out challenge не загружен. Это algorithmic synthetic benchmark, а не разрешение парных измерений у людей.

**Техническое воспроизведение, 14.09.2026.** Ноутбук сохраняет историческую постановку и ограничения. Повторное исполнение проверяет расчёты и отображение; оно не меняет научный статус ветки. Тяжёлые результаты читаются из сохранённых пакетов с исходными проверками целостности. Источники и конфигурации используются из этой папки проекта.

In [1]:
# Portable execution support; scientific sources remain in this checkout.
import os
import sys
from pathlib import Path
_start = Path(os.environ.get("BREATH_NOTEBOOK_DIR", Path.cwd())).resolve()
_support = next((folder for parent in (_start, *_start.parents)
                 for folder in (parent, parent / "notebooks", parent / "breath geometry/notebooks")
                 if (folder / "execution_support.py").is_file()), None)
if _support is None:
    raise FileNotFoundError("Не найден notebooks/execution_support.py")
sys.path.insert(0, str(_support))
from execution_support import roots, copdgene_case_dir, lungct_input
REPO_ROOT, ARTIFACT_ROOT = roots()
repo_root = REPO_ROOT


In [2]:
import hashlib
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import ndimage

RESULT_DIR = ARTIFACT_ROOT / "results/piecewise_svf_j12_contact_development_v3"
MANIFEST_PATH = RESULT_DIR / "manifest.json"
SUMMARY_PATH = RESULT_DIR / "summary.csv"
assert MANIFEST_PATH.is_file() and SUMMARY_PATH.is_file()

def sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest().upper()

manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
summary = pd.read_csv(SUMMARY_PATH)
print(f"code: {manifest['code_version']}")
print(f"runs: {manifest['run_count']}; selected_variant={manifest['selected_variant']}")
print(f"challenge_loaded={manifest['challenge']['loaded']}")

code: fcfeb1cbadf60a51903564348954e65344a5c812
runs: 9; selected_variant=None
challenge_loaded=False


In [3]:
checks = {
    "suite": sha256(REPO_ROOT / "configs" / manifest["suite_config"]["path"])
    == manifest["suite_config"]["sha256"],
    "search": sha256(REPO_ROOT / "configs" / manifest["search_config"]["path"])
    == manifest["search_config"]["sha256"],
    "summary": sha256(SUMMARY_PATH) == manifest["summary_sha256"],
}
for name, expected in manifest["field_sha256"].items():
    checks[f"field/{name}"] = sha256(RESULT_DIR / name) == expected
for name, expected in manifest["record_sha256"].items():
    checks[f"record/{name}"] = sha256(RESULT_DIR / name) == expected
assert all(checks.values()), {key: value for key, value in checks.items() if not value}
print(f"checksum PASS: {sum(checks.values())}/{len(checks)}")
pd.Series(manifest["variant_pass_counts"], name="passed_cases").to_frame()

AssertionError: {'field/contact_strong__dev_contact_counter_rotation.npz': False}

In [ ]:
columns = [
    "variant_id",
    "case_id",
    "endpoint_p95_limit_mm",
    "lung_field_p95_mm",
    "body_field_p95_mm",
    "tangential_slip_truth_median_mm",
    "tangential_slip_observed_median_mm",
    "tangential_slip_error_mm",
    "advected_target_surface_p95_max_mm",
    "advected_surface_coverage_min",
    "lung_jacobian_p01",
    "body_jacobian_p01",
    "gate_pass",
    "gate_reasons",
]
summary[columns].style.format({
    name: "{:.3f}"
    for name in columns
    if name.endswith("_mm") or name.endswith("_min") or name.endswith("_p01")
})

In [ ]:
labels = summary["variant_id"] + "\n" + summary["case_id"].str.replace(
    "dev_contact_", "", regex=False
)
x = np.arange(len(summary))
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
lung_relative = summary["lung_field_p95_mm"] / summary["endpoint_p95_limit_mm"]
body_relative = summary["body_field_p95_mm"] / summary["endpoint_p95_limit_mm"]
axes[0].bar(x - 0.2, lung_relative, 0.4, label="lung")
axes[0].bar(x + 0.2, body_relative, 0.4, label="body")
axes[0].axhline(1.0, color="black", linestyle="--", label="gate")
axes[0].set_ylabel("endpoint p95 / case limit")
axes[0].legend()

axes[1].bar(x - 0.2, summary["tangential_slip_truth_median_mm"], 0.4, label="truth")
axes[1].bar(x + 0.2, summary["tangential_slip_observed_median_mm"], 0.4, label="estimated")
axes[1].set_ylabel("median interregional tangential slip, mm")
axes[1].legend()

axes[2].bar(x, summary["advected_target_surface_p95_max_mm"], color="C2")
axes[2].axhline(0.75, color="black", linestyle="--", label="surface gate")
axes[2].set_ylabel("advected target-surface p95, mm")
axes[2].legend()
for axis in axes:
    axis.set_xticks(x, labels, rotation=70, ha="right", fontsize=7)
    axis.grid(axis="y", alpha=0.25)
fig.suptitle("Contact passes, but regional correspondence collapses to near-glued motion")
fig.tight_layout()
plt.show()

In [ ]:
artifact = np.load(RESULT_DIR / "balanced__dev_contact_counter_rotation.npz")
fixed_lung = artifact["fixed_lung_mask"].astype(bool)
surface = fixed_lung & ~ndimage.binary_erosion(fixed_lung)
truth_delta = (
    artifact["truth_lung_displacement_mm"] - artifact["truth_body_displacement_mm"]
)
estimated_delta = artifact["lung_displacement_mm"] - artifact["body_displacement_mm"]
truth_magnitude = np.linalg.norm(truth_delta, axis=-1)
estimated_magnitude = np.linalg.norm(estimated_delta, axis=-1)
error_magnitude = np.linalg.norm(estimated_delta - truth_delta, axis=-1)
z = fixed_lung.shape[2] // 2
maps = [truth_magnitude, estimated_magnitude, error_magnitude]
titles = ["truth regional jump", "estimated regional jump", "jump endpoint error"]
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for axis, values, title in zip(axes, maps, titles, strict=True):
    shown = np.where(surface[:, :, z], values[:, :, z], np.nan)
    image = axis.imshow(shown.T, origin="lower", cmap="magma", vmin=0, vmax=3.5)
    axis.set_title(title)
    axis.set_aspect("equal")
    axis.set_xticks([])
    axis.set_yticks([])
fig.colorbar(image, ax=axes, label="mm", shrink=0.8)
fig.suptitle("Balanced counter-rotation: interface jump on the central axial slice")
plt.show()

In [ ]:
runtime = summary[[
    "variant_id",
    "case_id",
    "objective_initial",
    "objective_final",
    "rejected_topology_updates",
    "elapsed_s",
    "peak_gpu_memory_bytes",
]].copy()
runtime["peak_gpu_memory_mib"] = runtime["peak_gpu_memory_bytes"] / 2**20
runtime.drop(columns="peak_gpu_memory_bytes").style.format({
    "objective_initial": "{:.3f}",
    "objective_final": "{:.3f}",
    "elapsed_s": "{:.2f}",
    "peak_gpu_memory_mib": "{:.1f}",
})

In [ ]:
assert len(summary) == 9
assert not summary["gate_pass"].any()
assert manifest["selected_variant"] is None
assert not manifest["candidate_freeze_allowed"]
assert not manifest["challenge"]["loaded"]
observed_slip_max = summary["tangential_slip_observed_median_mm"].max()
truth_slip_min = summary["tangential_slip_truth_median_mm"].min()
print("J1.2 torch-v0 DEVELOPMENT FAIL 0/9")
print(f"truth slip >= {truth_slip_min:.3f} mm; estimated slip <= {observed_slip_max:.3f} mm")
print("contact and topology pass, but the optimizer converges to a near-glued solution")
print("no candidate selected; held-out challenge must remain closed")
print("next: diagnose data-term identifiability before defining a new algorithm version")